In [1]:
%%capture
!pip install -q sentence-transformers chromadb

In [2]:
%%capture

import pandas as pd
from sentence_transformers import SentenceTransformer, util
import chromadb
from chromadb.utils import embedding_functions

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('/content/imdb_top_1000_fixed.csv')

# Crear el texto que se llevara a la base de datos
columns_to_combine = ['Overview', 'Director', 'Star1', 'Star2', 'Star3', 'Star4']
df['text'] = df[columns_to_combine].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)
df['ids'] = df.index.astype('str')

# Crea los embeddings de la columna text
embeddings = model.encode(df['text'], batch_size=64, show_progress_bar=True)
df['embeddings'] = embeddings.tolist()

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

## Chroma desde embeddings

In [4]:
## Base de datos
chroma_client = chromadb.Client()
client_persistent = chromadb.PersistentClient(path='/content/data_embeddings')

# genera los embeddings usando el modelo all-MiniLM-L6-v2
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name = 'all-MiniLM-L6-v2')

## Crear una coleccion de la base de datos con el nombre - movies_db - y usara el sentence_transformer_ef para crear los emeddigns
db = client_persistent.create_collection(name='movies_db', embedding_function=sentence_transformer_ef)

# Agregar datos a la colleccion movies_db por embeddings (Esto es importantisimo)
db.add(
    ids = df['ids'].tolist(),
    embeddings = df['embeddings'].tolist(),
    metadatas = df.drop(['ids','embeddings','text'],axis=1).to_dict('records')
)

## Chroma desde documents

In [5]:
## Crear una coleccion de la base de datos con el nombre - movies_db_no_embeddings - y usara el sentence_transformer_ef para crear los emeddigns
db_no_embeddings = client_persistent.create_collection(name='movies_db_no_embeddings', embedding_function=sentence_transformer_ef)

# Agregar datos a la colleccion movies_db_no_embeddings por documentos (Esto es importantisimo)
db_no_embeddings.add(
    ids=df['ids'].tolist(),
    documents=df['text'].tolist(),
    metadatas= df.drop(['ids','embeddings','text'],axis=1).to_dict('records')
)

### Chroma Query

In [6]:
## Realizar una busqueda y retorne los 2 registros mas parecidos
results = db.query(
    query_texts=['a history with elves and a ring'],
    n_results=2
)
## Retonar el resultado de la bisqueda
results

{'ids': [['609', '10']],
 'embeddings': None,
 'documents': [[None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'IMDB_Rating': 7.8,
    'Star4': 'Ken Stott',
    'No_of_Votes': 601408,
    'Released_Year': '2013',
    'Runtime': '161 min',
    'Certificate': 'UA',
    'Director': 'Peter Jackson',
    'Star3': 'Richard Armitage',
    'Overview': 'The dwarves, along with Bilbo Baggins and Gandalf the Grey, continue their quest to reclaim Erebor, their homeland, from Smaug. Bilbo Baggins is in possession of a mysterious and magical ring.',
    'Poster_Link': 'https://m.media-amazon.com/images/M/MV5BMzU0NDY0NDEzNV5BMl5BanBnXkFtZTgwOTIxNDU1MDE@._V1_UX67_CR0,0,67,98_AL_.jpg',
    'Series_Title': 'The Hobbit: The Desolation of Smaug',
    'Meta_score': 66.0,
    'Star2': 'Martin Freeman',
    'Gross': '258,366,855',
    'Genre': 'Adventure, Fantasy',
    'Star1': 'Ian McKellen'},
   {'Gross': '315,544,750',
    'Star4': 'Sean Be

### Cargar índice de Chroma

In [7]:
## cargar conexion a la coleccion
client_persistent_2 = chromadb.PersistentClient(path="/content/data_embeddings")
db_2 = client_persistent_2.get_collection('movies_db_no_embeddings')

In [8]:
## Realizar una busqueda y retorne los 2 registros mas parecidos
results = db_2.query(
    query_texts=['a history with elves and a ring'],
    n_results=2
)
## Retonar el resultado de la bisqueda
results

{'ids': [['609', '10']],
 'embeddings': None,
 'documents': [['The dwarves, along with Bilbo Baggins and Gandalf the Grey, continue their quest to reclaim Erebor, their homeland, from Smaug. Bilbo Baggins is in possession of a mysterious and magical ring. Peter Jackson Ian McKellen Martin Freeman Richard Armitage Ken Stott',
   'A meek Hobbit from the Shire and eight companions set out on a journey to destroy the powerful One Ring and save Middle-earth from the Dark Lord Sauron. Peter Jackson Elijah Wood Ian McKellen Orlando Bloom Sean Bean']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'Released_Year': '2013',
    'Star3': 'Richard Armitage',
    'Director': 'Peter Jackson',
    'Runtime': '161 min',
    'Gross': '258,366,855',
    'Star2': 'Martin Freeman',
    'Overview': 'The dwarves, along with Bilbo Baggins and Gandalf the Grey, continue their quest to reclaim Erebor, their homeland, from Smaug. Bilbo Baggins is in possess